<a href="https://colab.research.google.com/github/nibaskumar93n-debug/Morphoinformatics/blob/main/scRNA-seq_to_count_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda3

import os
os.environ['PATH'] = '/usr/local/miniconda3/bin:' + os.environ['PATH']

!conda --version

PREFIX=/usr/local/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local/miniconda3
conda 26.3.2


In [5]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [6]:
!conda install -y -c bioconda star
!STAR --version

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / done
Channels:
 - bioconda
 - defaults
Platform: linux-64
Solving environment: / - done

## Package Plan ##

  environment location: /usr/local/miniconda3

  added / updated specs:
    - star


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    conda-26.3.2               |  py313h06a4308_1         1.2 MB
    htslib-1.21                |       h5efdd21_0         3.0 MB  bioconda
    libdeflate-1.22            |       h5eee18b_0          68 KB
    star-2.7.11b               |       h5ca1c30_6         8.8 MB  bioconda
    ------------------------------------------------------------
                                           Total:        13.1 MB

The following NEW packages will be INSTALLED:

  htslib             bioconda/linux-64::htslib-1.21-h5efdd21_0 
  libdeflate         pkgs/main/linux-64::lib

In [5]:
!bash /content/ena-file-download-selected-files-20260514-0907.sh

--2026-05-14 11:21:51--  ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR255/073/SRR25528573/SRR25528573_2.fastq.gz
           => ‘SRR25528573_2.fastq.gz’
Resolving ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)|193.62.193.165|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /vol1/fastq/SRR255/073/SRR25528573 ... done.
==> SIZE SRR25528573_2.fastq.gz ... 873030570
==> PASV ... done.    ==> RETR SRR25528573_2.fastq.gz ... done.
Length: 873030570 (833M) (unauthoritative)

SRR25528573_2.fastq 100%[===================>] 832.59M  14.8MB/s    in 56s     

2026-05-14 11:22:49 (14.7 MB/s) - ‘SRR25528573_2.fastq.gz’ saved [873030570]

--2026-05-14 11:22:49--  ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR255/073/SRR25528573/SRR25528573_1.fastq.gz
           => ‘SRR25528573_1.fastq.gz’
Resolving ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.sra.

In [6]:
import os
os.makedirs('/content/fastq_renamed', exist_ok=True)

# Key change: use SRR number as filename, sample name for renamed file
ena_samples = {
    'sample1': 'SRR25528574',
    'sample2': 'SRR25528573',
}

for i, (sample, srr) in enumerate(ena_samples.items(), 1):
    r1_old = f'/content/fastq_files/{srr}_1.fastq.gz'  # actual filename
    r2_old = f'/content/fastq_files/{srr}_2.fastq.gz'

    r1_new = f'/content/fastq_renamed/{sample}_S{i}_L001_R1_001.fastq.gz'
    r2_new = f'/content/fastq_renamed/{sample}_S{i}_L001_R2_001.fastq.gz'

    os.rename(r1_old, r1_new)
    os.rename(r2_old, r2_new)
    print(f"✓ {srr} renamed to {sample}")

print("✓ All files renamed!")

✓ SRR25528574 renamed to sample1
✓ SRR25528573 renamed to sample2
✓ All files renamed!


In [1]:
!mkdir -p /content/reference

print("Downloading genome...")
!wget -q -c https://ftp.ensembl.org/pub/release-109/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz \
    -O /content/reference/genome.fa.gz

print("Downloading GTF...")
!wget -q -c https://ftp.ensembl.org/pub/release-109/gtf/homo_sapiens/Homo_sapiens.GRCh38.109.gtf.gz \
    -O /content/reference/genome.gtf.gz

print("Unzipping...")
!gunzip -f /content/reference/genome.fa.gz
!gunzip -f /content/reference/genome.gtf.gz

print("✓ Done!")

Unzipping...
✓ Done!


In [ ]:
!mkdir -p /content/star_index

print("Building STAR index... 30-45 minutes...")

!STAR --runMode genomeGenerate \
    --genomeDir /content/star_index \
    --genomeFastaFiles /content/reference/genome.fa \
    --sjdbGTFfile /content/reference/genome.gtf \
    --runThreadN 4 \
    --genomeSAindexNbases 14

print("✓ STAR index built!")

Building STAR index... 30-45 minutes...
	/usr/local/miniconda3/bin/STAR-avx2 --runMode genomeGenerate --genomeDir /content/star_index --genomeFastaFiles /content/reference/genome.fa --sjdbGTFfile /content/reference/genome.gtf --runThreadN 4 --genomeSAindexNbases 14
	STAR version: 2.7.11b   compiled: 2025-04-26T21:36:55+0000 :/opt/conda/conda-bld/star_1745703265435/work/source
May 14 11:38:06 ..... started STAR run
May 14 11:38:06 ... starting to generate Genome files
May 14 11:38:49 ..... processing annotations GTF
May 14 11:39:07 ... starting to sort Suffix Array. This may take a long time...
May 14 11:39:22 ... sorting Suffix Array chunks and saving them to disk...
May 14 12:22:34 ... loading chunks from disk, packing SA...


In [20]:
import os
os.makedirs('/content/starsolo_output', exist_ok=True)

for i, (sample, srr) in enumerate(ena_samples.items(), 1):

    r1 = f'/content/fastq_renamed/{sample}_S{i}_L001_R1_001.fastq.gz'
    r2 = f'/content/fastq_renamed/{sample}_S{i}_L001_R2_001.fastq.gz'

    print(f"\nRunning STARsolo for {sample} ({srr})...")

    !STAR --runMode alignReads \
        --genomeDir /content/star_index \
        --readFilesIn {r2} {r1} \
        --readFilesCommand zcat \
        --soloType CB_UMI_Simple \
        --soloCBwhitelist None \
        --soloCBstart 1 --soloCBlen 16 \
        --soloUMIstart 17 --soloUMIlen 12 \
        --outSAMtype BAM SortedByCoordinate \
        --outSAMattributes NH HI nM AS CR UR CB UB GX GN sS sQ sM \
        --outFileNamePrefix /content/starsolo_output/{sample}_ \
        --runThreadN 4 \
        --soloFeatures Gene \
        --soloCellFilter EmptyDrops_CR

    print(f"✓ {sample} done!")

print("\n✓ All samples processed!")


Running STARsolo for sample1 (SRR25528574)...
	/usr/local/miniconda3/bin/STAR-avx2 --runMode alignReads --genomeDir /content/star_index --readFilesIn /content/fastq_renamed/sample1_S1_L001_R2_001.fastq.gz /content/fastq_renamed/sample1_S1_L001_R1_001.fastq.gz --readFilesCommand zcat --soloType CB_UMI_Simple --soloCBwhitelist None --soloCBstart 1 --soloCBlen 16 --soloUMIstart 17 --soloUMIlen 12 --outSAMtype BAM SortedByCoordinate --outSAMattributes NH HI nM AS CR UR CB UB GX GN sS sQ sM --outFileNamePrefix /content/starsolo_output/sample1_ --runThreadN 4 --soloFeatures Gene --soloCellFilter EmptyDrops_CR
	STAR version: 2.7.11b   compiled: 2025-04-26T21:36:55+0000 :/opt/conda/conda-bld/star_1745703265435/work/source
May 14 09:32:59 ..... started STAR run
May 14 09:32:59 ..... loading genome

EXITING because of FATAL ERROR: could not open genome file /content/star_index//genomeParameters.txt
SOLUTION: check that the path to genome files, specified in --genomeDir is correct and the files 

In [ ]:
for sample in sample_list:
    print(f"\nOutput files for {sample}:")
    !ls /content/{sample}_output/outs/filtered_feature_bc_matrix/

In [ ]:
from google.colab import files
import shutil

for sample in sample_list:
    matrix_dir = f'/content/{sample}_output/outs/filtered_feature_bc_matrix'
    zip_path = f'/content/{sample}_matrix'

    shutil.make_archive(zip_path, 'zip', matrix_dir)
    files.download(f'{zip_path}.zip')
    print(f"✓ {sample} matrix downloaded!")

print("\n✓ All done!")